# VQPAINT token scorer (tokens -> MobileCLIP image embedding)

Distils teacher decoder + MobileCLIP-S0 vision into a small conv net on the token grid. Inputs: `vqpaint-assets` (checkpoint), `vqpaint-tokens` (grids + real strokes). Output: `/kaggle/working/scorer/`.

In [ ]:
S = dict(HOURS=2.5, BATCH=32, VARIANTS="S:64:64,96,128,192;M:64:96,128,192,256", BRANCH="research/tiny-vqgan",
         RESUME_FROM="/kaggle/input/vqpaint-tiny-scorer*/scorer/ckpt.pt")
import glob, os, shutil, subprocess, sys, time
TMP, WORK = "/kaggle/tmp", "/kaggle/working"; os.makedirs(TMP, exist_ok=True)
REPO, DATA, OUT = f"{TMP}/VQPAINT", f"{TMP}/data", f"{WORK}/scorer"
def sh(cmd):
    print("$", cmd, flush=True); subprocess.run(cmd, shell=True, check=True)
print(S)

In [ ]:
if not os.path.isdir(REPO): sh(f"git clone -q --branch {S['BRANCH']} --depth 1 https://github.com/NazarGol/VQPAINT.git {REPO}")
TAM = f"{REPO}/web/export/taming-transformers"
if not os.path.isdir(TAM): sh(f"git clone -q --depth 1 https://github.com/CompVis/taming-transformers.git {TAM}")
ASSETS = next(iter(glob.glob("/kaggle/input/vqpaint-assets*")), None); assert ASSETS
CK = f"{REPO}/web/export/checkpoints"
if not os.path.isdir(CK): os.symlink(f"{ASSETS}/checkpoints", CK)
TOK = next(iter(glob.glob("/kaggle/input/vqpaint-tokens*")), None); assert TOK
if not os.path.isdir(DATA): shutil.copytree(TOK, DATA)
sh(f"{sys.executable} -m pip install -q tokenizers scipy")
sh(f"{sys.executable} -m pip install -q onnxruntime-gpu || {sys.executable} -m pip install -q onnxruntime")
for f in ("onnx/vision_model.onnx", "onnx/text_model.onnx", "tokenizer.json"):
    dst = f"{DATA}/{os.path.basename(f)}"
    if not os.path.exists(dst): sh(f"curl -sSL https://huggingface.co/Xenova/mobileclip_s0/resolve/main/{f} -o {dst}")
sh(f"ls -la {DATA} && nvidia-smi --query-gpu=name,memory.total --format=csv")
import onnxruntime as ort; print("ORT providers:", ort.get_available_providers())

In [ ]:
os.makedirs(OUT, exist_ok=True)
prev = sorted(glob.glob(S["RESUME_FROM"])) if S["RESUME_FROM"] else []
if prev and not os.path.exists(f"{OUT}/ckpt.pt"): shutil.copy(prev[-1], f"{OUT}/ckpt.pt"); print("resuming from", prev[-1])

In [ ]:
cmd = f"{sys.executable} -u {REPO}/web/research/tiny/train_scorer.py --data {DATA} --out {OUT} --hours {S['HOURS']} --batch {S['BATCH']} --variants '{S['VARIANTS']}' --clip-vision {DATA}/vision_model.onnx --clip-text {DATA}/text_model.onnx --tokenizer {DATA}/tokenizer.json"
print("$", cmd, flush=True)
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
with open(f"{OUT}/train.log", "a") as log:
    for line in p.stdout:
        if "Warning" in line or "warn(" in line: continue
        print(line, end="", flush=True); log.write(line)
print("exit", p.wait())

In [ ]:
import json; print(json.dumps(json.load(open(f"{OUT}/eval.json")), indent=1)); sh(f"ls -la {OUT}")